In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torch transformers sentence-transformers -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# ── The core idea ─────────────────────────────────────────────────────────────
# Standard language: Agent A → tokenize → decode text → Agent B reads text
# Dense protocol:    Agent A → embedding vector → Agent B reads vector directly
#
# We benchmark: how much information (bits) is transmitted per "token" in each?
# And: can a human auditor understand what was communicated in the dense channel?

embedder = SentenceTransformer("LaBSE")

# ── Shared task: referential communication ────────────────────────────────────
# Agent A sees a concept. Agent B must identify it from N candidates.
# In text mode: A sends a description in words.
# In dense mode: A sends a latent vector directly.
# We measure: accuracy, message size (bytes), bits per correct identification.

CONCEPTS = [
    "saudade — longing for something absent",
    "schadenfreude — pleasure at another's misfortune",
    "hygge — cozy contentment",
    "ubuntu — shared humanity",
    "wabi-sabi — beauty in imperfection",
    "fernweh — longing to travel far away",
    "meraki — putting your soul into your work",
    "mamihlapinatapai — shared unspoken desire",
    "aware — bittersweet impermanence",
    "torschlusspanik — fear that time is running out",
]

# ── Text protocol ─────────────────────────────────────────────────────────────
def text_protocol_identify(target_idx, candidates, model):
    """Agent A sends text description. Agent B embeds it and picks closest."""
    target_text = candidates[target_idx]
    # Message = the text itself (this is what gets "sent")
    message_bytes = len(target_text.encode("utf-8"))
    
    # Agent B: embed all candidates, find closest to received text
    candidate_embs = model.encode(candidates, normalize_embeddings=True)
    message_emb    = model.encode([target_text], normalize_embeddings=True)
    sims = candidate_embs @ message_emb.T
    predicted_idx = int(sims.argmax())
    
    return predicted_idx == target_idx, message_bytes

# ── Dense protocol ────────────────────────────────────────────────────────────
def dense_protocol_identify(target_idx, candidates, model, noise_std=0.0):
    """Agent A sends embedding vector directly. Agent B finds closest candidate."""
    all_embs    = model.encode(candidates, normalize_embeddings=True)
    target_emb  = all_embs[target_idx:target_idx+1]
    
    # Message = the embedding vector (float32)
    message_bytes = target_emb.nbytes  # 768 floats * 4 bytes = 3072 bytes raw
    # With float16 quantization (realistic for transmission):
    message_bytes_compressed = target_emb.astype(np.float16).nbytes  # 1536 bytes
    
    # Inject channel noise
    noisy = target_emb + np.random.randn(*target_emb.shape) * noise_std
    noisy /= np.linalg.norm(noisy)
    
    sims = all_embs @ noisy.T
    predicted_idx = int(sims.argmax())
    
    return predicted_idx == target_idx, message_bytes_compressed

# ── Run comparison ────────────────────────────────────────────────────────────
N_TRIALS = 100
text_accs, text_sizes = [], []
dense_accs, dense_sizes = [], []

rng = np.random.default_rng(42)
for _ in range(N_TRIALS):
    target = rng.integers(0, len(CONCEPTS))
    t_acc, t_size = text_protocol_identify(target, CONCEPTS, embedder)
    d_acc, d_size = dense_protocol_identify(target, CONCEPTS, embedder, noise_std=0.0)
    text_accs.append(t_acc);  text_sizes.append(t_size)
    dense_accs.append(d_acc); dense_sizes.append(d_size)

text_acc_mean  = np.mean(text_accs)
dense_acc_mean = np.mean(dense_accs)
text_size_mean  = np.mean(text_sizes)
dense_size_mean = np.mean(dense_sizes)

# Information efficiency: bits of information per byte transmitted
# I(correct identification from N candidates) = log2(N) bits
info_bits = np.log2(len(CONCEPTS))
text_efficiency  = (text_acc_mean  * info_bits) / text_size_mean
dense_efficiency = (dense_acc_mean * info_bits) / dense_size_mean

print(f"{'Protocol':<12} {'Accuracy':>10} {'Msg size':>12} {'Bits/byte':>12}")
print("-"*50)
print(f"{'Text':<12} {text_acc_mean:>10.3f} {text_size_mean:>10.1f}B {text_efficiency:>12.6f}")
print(f"{'Dense':<12} {dense_acc_mean:>10.3f} {dense_size_mean:>10.1f}B {dense_efficiency:>12.6f}")
print(f"\nDense protocol is {dense_efficiency/text_efficiency:.1f}x more information-dense")

# ── Noise robustness ──────────────────────────────────────────────────────────
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
dense_noise_accs = []
for noise in noise_levels:
    accs = [dense_protocol_identify(rng.integers(0, len(CONCEPTS)), CONCEPTS, embedder, noise)[0]
            for _ in range(50)]
    dense_noise_accs.append(np.mean(accs))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Text protocol\n(full description)", "Dense protocol\n(float16 embedding)"],
            [text_efficiency, dense_efficiency], color=["#378ADD","#1D9E75"])
axes[0].set_ylabel("Bits transmitted per byte sent")
axes[0].set_title("Information density: dense >> text")

axes[1].plot(noise_levels, dense_noise_accs, marker="o", color="#1D9E75")
axes[1].axhline(y=text_acc_mean, color="#378ADD", linestyle="--", label="text protocol baseline")
axes[1].set_xlabel("Gaussian noise σ on embedding channel")
axes[1].set_ylabel("Identification accuracy")
axes[1].set_title("Dense protocol noise robustness")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("dense_communication_protocol.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nWhat to build next for this notebook:")
print("1. Train a lightweight encoder that compresses the 768-dim vector to 64-dim")
print("   while preserving identification accuracy — find the minimum channel width")
print("2. Add interpretability layer: for each transmitted vector, decode back to")
print("   nearest words in multiple languages = human-auditable message")
print("3. Multi-hop: A→B→C with degradation at each hop (text vs dense)")